In [24]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# args_schema, Tool

# args_schema를 정의한다는 것은 도구가 받는 인자의 이름, 데이터 타입, 구조에 대해서 명확히 한다는 것 
# args_schema를 정의하지 않으면 언어 모델은 오직 도구의 이름과 설명(description)과 시그니처를 사용해서 파악

# def get_weather_func(city: str) -> str:
# 시그니처와 args_schema는 본질적으로 같은 것이지만 함수 시그니처는 Python 함수 정의 형태로 얻는 정보이고
# args_schema는 Pydantic BaseModel 객체 형태

# -------------------------------------------------------------------------------------

# args_schema가 만들어지는 케이스

# 1. @tool 데코레이터 사용
# ---> 데코레이터가 내부적으로 생성한 StructuredTool 객체의 args_schema 속성에 자동으로 만든 args_schema를 직접 할당
# 2. StructuredTool.from_function() 사용: 기존 함수를 StructuredTool로 변환하면서 자동으로 args_schema 생성
# 역시 args_schema 속성에 자동으로 만든 args_schema를 직접 할당 
# 3. Pydantic의 BaseModel을 상속하는 클래스를 만들어서 Tool 객체의 args_schema로 할당


# Tool은 args_schema를 만들어서 args_schema 속성에 전달
# 전달 안하면? 함수 시그니처를 사용

# -------------------------------------------------------------------------------------

# 큰 그림은
# 1. args_schema -> 도구 스키마 형태로 언어 모델로 전달 
# 2. args_schema -> Tool에 전달
# 
# 3. 언어 모델의 도구 호출 지시를 Agent는 이를 AgentAction 형태로 바꾸어 AgentExecutor에게 전달
# 4. AgentExecutor는 전달받은 AgentAction의 도구 호출에 있는 'tool_name', 'tool_input'을 확인
# 5. AgentExecutor는 호출하려는 도구에 해당되는 Tool 객체를 알아낸 다음
# 6. Tool 객체의 args_schema(또는 함수 시그니처)를 사용해서 전달 받은 tool_input이 유효한 형식인지 검사
# Tool 객체에 args_schema가 없는 경우에는 함수 시그니처 기반으로 검증
# 시그니처와 args_schema는 본질적으로 같은 것이지만 함수 시그니처는 Python 함수 정의 형태로 얻는 정보이고
# args_schema는 Pydantic BaseModel 객체 형태

# 7. 유효하다면 AgentExecutor가 도구를 호출한 다음 리턴 값을 observation으로 변환

# -------------------------------------------------------------------------------------

# Tool 클래스를 직접 사용하는 이유
# 1. return_direct=True와 같은 고급 설정: 도구 호출의 결과를 Agent에게 보내지 않고 사용자로 바로 전달
# 2. 실행 시점에 동적으로 인자 스키마를 구성
# search(product_name: str, category: str) 함수의 경우 category가 계속 새로 추가될 수 있음

# 3. @tool 데코레이터를 붙일 수 없는 외부 라이브러리의 함수를 Tool로 래핑하여 사용 > @tool은 소스 코드가 있어야 함

In [25]:
# args_schema가 만들어지는 케이스 - 1

# args_schema는 @tool 데코레이터 사용으로 자동으로 생성한 다음
# 데코레이터가 내부적으로 생성한 StructuredTool 객체의 args_schema 속성에 직접 할당

from langchain_core.tools import tool

# @tool 데코레이터가 get_weather 함수를 StructuredTool 객체로 변환
@tool
def get_weather_func(city: str) -> str:
    """
    특정 도시의 현재 날씨를 가져오는 도구입니다.

    Args:
        city: 날씨를 알고 싶은 도시의 이름입니다.
    """
    return f"{city} 날씨는 맑음입니다."

print(get_weather_func.args_schema.model_json_schema())
# model_json_schema(): Pydantic 모델인 args_schema를 JSON Schema 형태로 변환



{'description': '특정 도시의 현재 날씨를 가져오는 도구입니다.\n\nArgs:\n    city: 날씨를 알고 싶은 도시의 이름입니다.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_func', 'type': 'object'}


In [22]:
# args_schema가 만들어지는 케이스 - 2

# 기존 함수를 StructuredTool.from_function()을 사용해서 StructuredTool로 변환하면서 자동으로 args_schema 생성
from langchain.tools import StructuredTool

def get_weather_func(city: str) -> str:
    """
    특정 도시의 현재 날씨를 가져오는 도구입니다.

    Args:
        city: 날씨를 알고 싶은 도시의 이름입니다.
    """
    return f"{city} 날씨는 맑음입니다."

weather_tool = StructuredTool.from_function(
    func=get_weather_func,
    name="get_weather",
    description="특정 도시의 현재 날씨를 가져오는 도구",
)

print(weather_tool.args_schema.model_json_schema())

{'description': '특정 도시의 현재 날씨를 가져오는 도구입니다.\n\nArgs:\n    city: 날씨를 알고 싶은 도시의 이름입니다.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather', 'type': 'object'}


In [23]:
# args_schema가 만들어지는 케이스 - 3

# Tool 생성자의 args_schema 매개변수로 Pydantic BaseModel 할당

from langchain.tools import Tool
from pydantic import BaseModel, Field

def get_weather_func(city: str) -> str:
    """
    특정 도시의 현재 날씨를 가져오는 도구입니다.

    Args:
        city: 날씨를 알고 싶은 도시의 이름입니다.
    """
    return f"{city} 날씨는 맑음입니다."

class WeatherInput(BaseModel):
    city: str = Field(description="날씨를 알고 싶은 도시의 이름")

weather_tool_explicit = Tool(
    name="get_weather",
    func=get_weather_func,
    description="...",
    args_schema=WeatherInput
)

print(weather_tool_explicit.args_schema.model_json_schema())

{'properties': {'city': {'description': '날씨를 알고 싶은 도시의 이름', 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'WeatherInput', 'type': 'object'}


In [ ]:
# 각 경우별 weather_tool_explicit.args_schema.model_json_schema()의 출력 결과
# @tool 데코레이터와 StructuredTool.from_function()의 경우는 동일함

def get_weather_func(city: str) -> str:
    """
    특정 도시의 현재 날씨를 가져오는 도구입니다.

    Args:
        city: 날씨를 알고 싶은 도시의 이름입니다.
    """
    return f"{city} 날씨는 맑음입니다."
# -------------------------------------------------------------------------------

# @tool 데코레이터
{
    'description': '특정 도시의 현재 날씨를 가져오는 도구입니다.\n\n'
    'Args:\n    '
    'city: 날씨를 알고 싶은 도시의 이름입니다.', 
    'properties': {'city': {'title': 'City', 'type': 'string'}}, 
    'required': ['city'], 
    'title': 'get_weather_func', 
    'type': 'object'
}

# StructuredTool.from_function()
{
    'description': '특정 도시의 현재 날씨를 가져오는 도구입니다.\n\n'
    'Args:\n    '
    'city: 날씨를 알고 싶은 도시의 이름입니다.', 
    'properties': {'city': {'title': 'City', 'type': 'string'}}, 
    'required': ['city'], 
    'title': 'get_weather', 
    'type': 'object'
}

# Pydantic BaseModel
{
    'properties': {'city': {'description': '날씨를 알고 싶은 도시의 이름', 'title': 'City', 'type': 'string'}}, 
    'required': ['city'], 
    'title': 'WeatherInput', 
    'type': 'object'
}
